In [ ]:
import os
import itertools
import requests
import rasterio
import pandas as pd

In [ ]:
# Parameter options
versions = ["V1", "V2", "V3"]
years = ["2021", "2024"]
countries = ["VNM", "ZAF"] # ["VNM", "COL", "ZAF", "CZE", "NOR", "GRC"]
types = ["coastalFlats", "mangroveExtent", "ocean",
"GlobalGrassland", "alpineRegions", "aridity", "desertClass", 
"extent", "islands", "k1Class", "landCover", "latitude",
"meanInterval", "ocean", "plantationYear", "seaforms",
"smod", "soilDepth", "streamPower", "subantarcticIslands",
"submergedMask", "t2maveragemax", "t2maveragemin", "tmaxP0",
"tminP0", "tpaveragemax", "tpaveragemin", "waterClass"]
# Base URL
base_url = (
    "https://s3.waw4-1.cloudferro.com/ecdc-waw4-1-ekqouvq3otv8hmw0njzuvo0g4dy0ys8r985n7dggjis3erkpn5o/"
    "preprocessed/global/Globes/{version}/Globes_{version}00_{year}_{country}_{type}.tiff"
)

# Output DataFrame
results = []

# Output directory (can be temporary)
output_dir = "temp_tiffs"
os.makedirs(output_dir, exist_ok=True)

# Loop through all combinations
for version, year, country, typ in itertools.product(versions, years, countries, types):
    url = base_url.format(version=version, year=year, country=country, type=typ)
    local_file = os.path.join(output_dir, f"{version}_{year}_{country}_{typ}.tiff")
    
    try:
        print(f"Downloading: {url}", end="\r", flush=True)
        response = requests.get(url, stream=True)
        if response.status_code != 200:
            raise ValueError("File not found")

        with open(local_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        # Open with rasterio and check if all data is nodata
        with rasterio.open(local_file) as src:
            data = src.read(1, masked=True)
            all_nodata = data.mask.all()
        
        if all_nodata:
            all_nodata = "Okay"
        else:
            all_nodata = "No_data"
        
        results.append({
            "version": version,
            "year": year,
            "country": country,
            "type": typ,
            "URL": url,
            "status": all_nodata
        })

    except Exception as e:
        print(f"Error with {url}: {e}", end="\r", flush=True)
        results.append({
            "version": version,
            "year": year,
            "country": country,
            "type": typ,
            "URL": url,
            "status": str(e)
        })

    finally:
        if os.path.exists(local_file):
            os.remove(local_file)

# Remove the temp folder

# Make sure that is empthy before
for filename in os.listdir(output_dir):
    file_path = os.path.join(output_dir, filename)
    os.remove(file_path)

os.rmdir(output_dir)

# Create dataframe
df = pd.DataFrame(results)



In [9]:
df.to_csv("nodata_check_results.csv", index=False)